# Experience Replay

## Waste of Experience

- `transition`：$(s_t, a_t, r_t, s_{t+1})$；
- `experience`：所有的`transition`，针对$t=1, t=2,...$；
- `TD`算法中，用完一个`transition`，就将其丢弃，不再使用，造成对经验的浪费；
- 而事实上经验可以重复使用。

## Correlated Updates

- 按照顺序使用每一条`transition`来更新$w$；
- 前后两条`transition`有很强的关联（实验证明这种强关联是有害的，如果可以将序列打散，可以获得更好的效果）。

## Experience Replay

- 一条`transition`：$(s_t, a_t, r_t, s_{t+1})$，相当于一条训练数据；
- 将最近$n$条`transition`放入到一个回放队列（`replay buffer`）中；
- $n$为超参数，通常很大，在$10^5 \sim 10^6$之间，取决于具体应用。

- 训练`DQN`的目标为：$L(w) = \frac 1 T \sum_{t=1}^T \frac {\delta_t^2}{2}$
- 通过随机梯度下降（SGD）：
    - 每次从`buffer`中抽取一个`transition`
    - 计算`TD error`$\delta_i$
    - 随机梯度下降$g_i = \frac {\partial \delta_i^2 / 2}  {\partial w} = \delta_i \cdot \frac {\partial Q(s_i, a_i; w)}{\partial w}$
    - SGD：$w \leftarrow w - \alpha \cdot g_i$

打破了序列的相关性，并且可以重复使用过去的经验。

## Prioritized Experience Replay

采用非均匀抽样代替均匀抽样。（例如《超级马里奥》中打boss的经验或跑图的经验）

如何确定`transition`的重要性？可以通过`TD error`$|\delta_t|$来判断，绝对值越大则越重要。

- 抽样概率：$p_t \propto |\delta_t| + \epsilon $
- 抽样概率：$p_t \propto |\delta_t| + \frac 1 {rank(t)} $

如果做均匀抽样，所有的`transition`都应该有相同的学习率；如果做分均匀抽样，应该根据抽样概率来调整学习率。
- 如果一条`transition`有较大的抽样概率，应该将其学习率设置的比较小，计算$(n p_t)^{-\beta}, \beta \in (0, 1)$
- $\beta$为超参数，其实很小，逐渐增大，直至$1$

- 每条`transition`需要标记一个`TD error`$\delta_t$；
- 如果这条`transition`刚刚收集到，不知道$\delta_t$，则设置为最大值。
- 训练`DQN`时，需要对`buffer`做更新。